In [ ]:
# ============================================================
# WEEK 3 - MACHINE LEARNING MODEL DEVELOPMENT AND EVALUATION
# Project: Customer Churn Prediction
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Libraries imported successfully!")


# ------------------------------------------------------------
# 2. CREATE PROJECT FOLDERS
# ------------------------------------------------------------

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Project folders are ready.")


# ------------------------------------------------------------
# 3. CREATE SAMPLE DATASET
# ------------------------------------------------------------
# No real dataset was provided for this task.
# Therefore, a sample customer churn dataset is created.
# If you have a real CSV later, you can replace this section.

def create_sample_dataset(n=1500):

    np.random.seed(RANDOM_STATE)

    data = {
        "age": np.random.randint(18, 70, n),

        "tenure_months": np.random.randint(1, 73, n),

        "monthly_charges": np.round(
            np.random.uniform(20, 150, n), 2
        ),

        "total_usage_hours": np.round(
            np.random.uniform(1, 300, n), 2
        ),

        "support_calls": np.random.randint(
            0, 12, n
        ),

        "payment_delays": np.random.randint(
            0, 6, n
        ),

        "contract_type": np.random.choice(
            ["Monthly", "Yearly", "Two-Year"],
            n,
            p=[0.55, 0.30, 0.15]
        ),

        "internet_service": np.random.choice(
            ["Basic", "Standard", "Premium"],
            n
        ),

        "payment_method": np.random.choice(
            [
                "Credit Card",
                "Debit Card",
                "UPI",
                "Bank Transfer"
            ],
            n
        )
    }

    df = pd.DataFrame(data)

    # Create a simple churn score
    churn_score = (
        0.8 * (df["support_calls"] / 12)
        + 0.8 * (df["payment_delays"] / 5)
        + 0.7 * (df["monthly_charges"] / 150)
        - 0.7 * (df["tenure_months"] / 72)
        + 0.3 * (df["contract_type"] == "Monthly")
        + np.random.normal(0, 0.25, n)
    )

    # Convert score into probability
    probability = 1 / (
        1 + np.exp(-churn_score + 1.1)
    )

    # Create target variable
    df["churn"] = (
        np.random.random(n) < probability
    ).astype(int)

    # Add some missing values
    for column in [
        "monthly_charges",
        "total_usage_hours",
        "payment_method"
    ]:

        indexes = np.random.choice(
            df.index,
            size=int(n * 0.03),
            replace=False
        )

        df.loc[indexes, column] = np.nan

    return df


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

real_dataset = "data/customer_churn.csv"

if os.path.exists(real_dataset):

    print("Real dataset found.")
    df = pd.read_csv(real_dataset)

else:

    print("No real dataset found.")
    print("Creating sample customer churn dataset...")

    df = create_sample_dataset()

    df.to_csv(
        "data/sample_customer_churn.csv",
        index=False
    )


print("\nDataset created successfully.")


# ------------------------------------------------------------
# 5. BASIC DATA EXPLORATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nStatistical Summary:")
display(df.describe(include="all"))


# ------------------------------------------------------------
# 6. REMOVE DUPLICATES
# ------------------------------------------------------------

df = df.drop_duplicates()

print("\nDataset shape after removing duplicates:")
print(df.shape)


# ------------------------------------------------------------
# 7. DEFINE FEATURES AND TARGET
# ------------------------------------------------------------

target = "churn"

X = df.drop(columns=[target])

y = df[target]

print("\nTarget variable:", target)

print("\nTarget distribution:")
print(y.value_counts())


# ------------------------------------------------------------
# 8. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ------------------------------------------------------------

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


# ------------------------------------------------------------
# 9. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ------------------------------------------------------------
# 10. DATA PREPROCESSING
# ------------------------------------------------------------

# Numerical features:
# Missing values -> median
# Scaling -> StandardScaler

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# Categorical features:
# Missing values -> most frequent
# Categories -> One Hot Encoding

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# Combine both pipelines

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("\nPreprocessing pipeline created.")


# ------------------------------------------------------------
# 11. DEFINE MACHINE LEARNING MODELS
# ------------------------------------------------------------

models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            max_depth=6,
            random_state=RANDOM_STATE
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=RANDOM_STATE
        )
}


# ------------------------------------------------------------
# 12. TRAIN MODELS
# ------------------------------------------------------------

trained_models = {}

results = []

for name, model in models.items():

    print("\n" + "=" * 60)
    print("TRAINING:", name)
    print("=" * 60)

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    # Train model
    pipeline.fit(
        X_train,
        y_train
    )

    # Predictions
    y_pred = pipeline.predict(
        X_test
    )

    # Probability predictions
    y_probability = pipeline.predict_proba(
        X_test
    )[:, 1]

    # Evaluation metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_probability
    )

    # Store results
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    })

    trained_models[name] = pipeline

    print("\nAccuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC AUC  :", round(roc_auc, 4))


# ------------------------------------------------------------
# 13. MODEL COMPARISON
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

display(results_df)

results_df.to_csv(
    "results/model_comparison.csv",
    index=False
)


# ------------------------------------------------------------
# 14. CROSS-VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION")
print("=" * 60)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = []

for name, pipeline in trained_models.items():

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"
    )

    mean_score = scores.mean()
    std_score = scores.std()

    cv_results.append({
        "Model": name,
        "Mean F1": mean_score,
        "Std F1": std_score
    })

    print(
        f"{name}: "
        f"Mean F1 = {mean_score:.4f}, "
        f"Std = {std_score:.4f}"
    )


cv_df = pd.DataFrame(cv_results)

display(cv_df)

cv_df.to_csv(
    "results/cross_validation_results.csv",
    index=False
)


# ------------------------------------------------------------
# 15. HYPERPARAMETER TUNING
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING - RANDOM FOREST")
print("=" * 60)


rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE
            )
        )
    ]
)


# Parameters to test

param_grid = {

    "model__n_estimators": [
        100,
        200
    ],

    "model__max_depth": [
        5,
        10,
        15,
        None
    ],

    "model__min_samples_split": [
        2,
        5,
        10
    ]
}


grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)


# IMPORTANT:
# This line must execute before using
# best_params_ or best_score_

grid_search.fit(
    X_train,
    y_train
)


print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation F1:")
print(round(grid_search.best_score_, 4))


# ------------------------------------------------------------
# 16. GET BEST MODEL
# ------------------------------------------------------------

best_model = grid_search.best_estimator_

print("\nBest model selected successfully.")


# ------------------------------------------------------------
# 17. FINAL MODEL PREDICTIONS
# ------------------------------------------------------------

final_predictions = best_model.predict(
    X_test
)

final_probabilities = best_model.predict_proba(
    X_test
)[:, 1]


# ------------------------------------------------------------
# 18. FINAL MODEL METRICS
# ------------------------------------------------------------

final_accuracy = accuracy_score(
    y_test,
    final_predictions
)

final_precision = precision_score(
    y_test,
    final_predictions,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_predictions,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    final_predictions,
    zero_division=0
)

final_auc = roc_auc_score(
    y_test,
    final_probabilities
)


print("\n" + "=" * 60)
print("FINAL MODEL RESULTS")
print("=" * 60)

print(
    f"Accuracy : {final_accuracy:.4f}"
)

print(
    f"Precision: {final_precision:.4f}"
)

print(
    f"Recall   : {final_recall:.4f}"
)

print(
    f"F1 Score : {final_f1:.4f}"
)

print(
    f"ROC AUC  : {final_auc:.4f}"
)


# ------------------------------------------------------------
# 19. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        final_predictions,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 20. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    final_predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot()

plt.title(
    "Customer Churn - Confusion Matrix"
)

plt.tight_layout()

plt.savefig(
    "results/confusion_matrix.png",
    dpi=200
)

plt.show()


# ------------------------------------------------------------
# 21. ROC CURVE
# ------------------------------------------------------------

RocCurveDisplay.from_predictions(
    y_test,
    final_probabilities
)

plt.title(
    "Customer Churn - ROC Curve"
)

plt.tight_layout()

plt.savefig(
    "results/roc_curve.png",
    dpi=200
)

plt.show()


# ------------------------------------------------------------
# 22. SAVE FINAL MODEL
# ------------------------------------------------------------

model_path = (
    "models/customer_churn_model.pkl"
)

joblib.dump(
    best_model,
    model_path
)

print("\nFinal model saved successfully:")
print(model_path)


# ------------------------------------------------------------
# 23. CREATE FINAL RESULTS FILE
# ------------------------------------------------------------

final_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC AUC"
    ],

    "Score": [
        final_accuracy,
        final_precision,
        final_recall,
        final_f1,
        final_auc
    ]
})

display(final_results)

final_results.to_csv(
    "results/final_model_results.csv",
    index=False
)


# ------------------------------------------------------------
# 24. SAMPLE CUSTOMER PREDICTION
# ------------------------------------------------------------

def predict_customer_churn(customer_data):

    probability = best_model.predict_proba(
        customer_data
    )[:, 1]

    prediction = (
        probability >= 0.50
    ).astype(int)

    output = customer_data.copy()

    output["churn_probability"] = (
        probability
    )

    output["predicted_churn"] = (
        prediction
    )

    return output


# Create one example customer

sample_customer = pd.DataFrame({

    "age": [25],

    "tenure_months": [8],

    "monthly_charges": [120],

    "total_usage_hours": [45],

    "support_calls": [8],

    "payment_delays": [3],

    "contract_type": ["Monthly"],

    "internet_service": ["Premium"],

    "payment_method": ["UPI"]
})


prediction = predict_customer_churn(
    sample_customer
)

print("\n" + "=" * 60)
print("SAMPLE CUSTOMER PREDICTION")
print("=" * 60)

display(prediction)


# ------------------------------------------------------------
# 25. FINAL PROJECT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("WEEK 3 PROJECT COMPLETED")
print("=" * 60)

print("""
Machine Learning workflow completed:

1. Dataset creation/loading
2. Data inspection
3. Duplicate removal
4. Feature and target selection
5. Train-test split
6. Missing value handling
7. Feature scaling
8. Categorical encoding
9. Logistic Regression
10. Decision Tree
11. Random Forest
12. Model comparison
13. 5-fold cross-validation
14. Hyperparameter tuning
15. Final model evaluation
16. Confusion matrix
17. ROC-AUC curve
18. Model saving
19. Sample prediction

The final model is ready for a future deployment stage.
""")